In [3]:
import os
import time
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
import joblib
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ========== Конфигурация ==========
MODEL_NAME = "ai-sage/GigaChat3-10B-A1.8B-bf16"
INPUT_CSV = "../data/bench/knowledge_bench_private_no_labels.csv"   # тестовый датасет с метками
ARTIFACTS_DIR = "../model/"  # папка с pca.pkl, scaler.pkl, hallucination_detector.pkl

PROBE_LAYERS = [0, 5, 10, 15, 20, 25]   # те же слои, что при обучении

# ========== Загрузка артефактов ==========
with open(ARTIFACTS_DIR + "pca.pkl", "rb") as f:
    pca = pickle.load(f)
with open(ARTIFACTS_DIR + "scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

xgb_model = joblib.load(ARTIFACTS_DIR + "hallucination_detector.pkl")
print("PCA, scaler и XGBoost загружены.")

# ========== Загрузка модели Gigachat (8-bit) ==========
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
    offload_folder="offload",
)
model.eval()
print("Gigachat загружена.")

PCA, scaler и XGBoost загружены.


config.json: 0.00B [00:00, ?B/s]

`rope_parameters`'s beta_fast field must be a float, got 32
`rope_parameters`'s beta_slow field must be a float, got 1


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/10.7M [00:00<?, ?B/s]

...

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

DeepseekV3ForCausalLM LOAD REPORT from: ai-sage/GigaChat3-10B-A1.8B-bf16
Key                                                 | Status     |  | 
----------------------------------------------------+------------+--+-
model.layers.26.self_attn.kv_b_proj.weight          | UNEXPECTED |  | 
model.layers.26.eh_proj.weight                      | UNEXPECTED |  | 
model.layers.26.mlp.shared_experts.gate_proj.weight | UNEXPECTED |  | 
model.layers.26.self_attn.q_proj.weight             | UNEXPECTED |  | 
model.layers.26.input_layernorm.weight              | UNEXPECTED |  | 
model.layers.26.mlp.gate.e_score_correction_bias    | UNEXPECTED |  | 
model.layers.26.hnorm.weight                        | UNEXPECTED |  | 
model.layers.26.self_attn.kv_a_proj_with_mqa.weight | UNEXPECTED |  | 
model.layers.26.shared_head.head.weight             | UNEXPECTED |  | 
model.layers.26.self_attn.o_proj.weight             | UNEXPECTED |  | 
model.layers.26.post_attention_layernorm.weight     | UNEXPECTED |  | 
mode

generation_config.json:   0%|          | 0.00/153 [00:00<?, ?B/s]

Gigachat загружена.


In [5]:
# ========== FeatureAccumulator (точно как при обучении) ==========
class FeatureAccumulator:
    def __init__(self, model, probe_layers):
        self.model = model
        self.probe_layers = probe_layers
        self._hooks = []
        self._hidden = {}
    def attach(self):
        self._hidden.clear()
        layers = self.model.model.layers
        for idx in self.probe_layers:
            if idx >= len(layers): continue
            name = f"layer_{idx}"
            def make_hook(n):
                def hook(_, __, out):
                    h = out[0] if isinstance(out, tuple) else out
                    self._hidden[n] = h.detach()
                return hook
            self._hooks.append(layers[idx].register_forward_hook(make_hook(name)))
    def detach(self):
        for h in self._hooks: h.remove()
        self._hooks.clear()
    def __enter__(self):
        self.attach()
        return self
    def __exit__(self, *args):
        self.detach()
    def compute_features(self, logits, input_ids, ans_start):
        seq_len = input_ids.shape[1]
        ans_len = seq_len - ans_start

        # --- Uncertainty features ---
        answer_logits = logits[0, ans_start-1:seq_len-1, :].float()
        answer_ids = input_ids[0, ans_start:seq_len]
        answer_ids = answer_ids.to(answer_logits.device)

        log_probs = F.log_softmax(answer_logits, dim=-1)
        token_lp = log_probs.gather(1, answer_ids.unsqueeze(1)).squeeze(-1)

        probs = F.softmax(answer_logits, dim=-1)
        entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=-1)
        top1 = probs.max(dim=-1).values
        top5 = probs.topk(min(5, probs.shape[-1]), dim=-1).values.sum(dim=-1)

        uncertainty = np.array([
            token_lp.mean().item(), token_lp.min().item(), token_lp.max().item(),
            token_lp.std().item() if ans_len > 1 else 0.0,
            entropy.mean().item(), entropy.max().item(),
            entropy.std().item() if ans_len > 1 else 0.0,
            torch.exp(-token_lp.mean()).item(), float(ans_len), token_lp[0].item(),
            top1.mean().item(), top5.mean().item()
        ], dtype=np.float32)

        # --- Internal features & probe vector ---
        last_hs = self._hidden[f"layer_{self.probe_layers[-1]}"][0]
        probe_vec = last_hs[ans_start-1].cpu().float().numpy()

        int_scalars = []
        for idx in self.probe_layers:
            hs = self._hidden[f"layer_{idx}"][0]
            int_scalars.append(hs[ans_start-1].norm().item())
            int_scalars.append(hs[ans_start:seq_len].norm(dim=-1).mean().item())

            ans_hs = hs[ans_start-1:seq_len-1].unsqueeze(0)
            with torch.no_grad():
                norm_hs = self.model.model.norm(ans_hs)
                ll = self.model.lm_head(norm_hs).float()
            ll_p = torch.softmax(ll[0], dim=-1)
            ll_e = -(ll_p * torch.log(ll_p + 1e-10)).sum(dim=-1)
            int_scalars.append(ll_e.mean().item())

        first_e, last_e = int_scalars[2], int_scalars[-1]
        int_scalars.append(first_e - last_e)
        int_scalars.append(last_e / (first_e + 1e-10))

        internal = np.array(int_scalars, dtype=np.float32)
        self._hidden.clear()
        return {"uncertainty": uncertainty, "internal_scalars": internal, "probe_vec": probe_vec}

In [6]:
# ========== Препроцессинг (точно как при обучении) ==========
def preprocess(tokenizer, prompt, answer):
    prompt_enc = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        tokenize=True
    )
    ans_start = len(prompt_enc["input_ids"])

    full_enc = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}, {"role": "assistant", "content": answer}],
        tokenize=True
    )
    tok_ids = torch.tensor([full_enc["input_ids"]], dtype=torch.long)
    return tok_ids, ans_start

# ========== Загрузка данных ==========
df_private = pd.read_csv(INPUT_CSV)
print(f"Загружено примеров: {len(df_private)}")

Загружено примеров: 1038


In [7]:
# ========== Инференс ==========
accumulator = FeatureAccumulator(model, PROBE_LAYERS)

t_model_list = []
t_overhead_list = []
pred_probas = []

for idx, row in tqdm(df_private.iterrows(), total=len(df_private), desc="Inference"):
    prompt = row["prompt"]
    answer = row["model_answer"]

    # 1. Токенизация
    tok, ans_start = preprocess(tokenizer, prompt, answer)
    tok = tok.to(model.device)

    # 2. Forward pass Gigachat (время не учитывается в overhead, но замерим для информации)
    t0 = time.perf_counter()
    with accumulator, torch.no_grad():
        out = model(tok)
    t_model = time.perf_counter() - t0

    # 3. Извлечение признаков и подготовка вектора для классификатора
    t1 = time.perf_counter()
    feats = accumulator.compute_features(out.logits, tok, ans_start)

    # Объединяем признаки как при обучении: probe + internal + uncertainty
    probe_vec = feats["probe_vec"].reshape(1, -1)
    internal = feats["internal_scalars"].reshape(1, -1)
    uncertainty = feats["uncertainty"].reshape(1, -1)

    # Применяем PCA к probe-части
    probe_pca = pca.transform(probe_vec)
    # Объединяем всё
    X = np.hstack([probe_pca, internal, uncertainty])
    # Применяем scaler
    X_scaled = scaler.transform(X)

    # 4. Предсказание XGBoost
    proba = xgb_model.predict_proba(X_scaled)[0, 1]  # вероятность класса 1 (галлюцинация)
    t_overhead = time.perf_counter() - t1

    pred_probas.append(proba)
    t_model_list.append(t_model * 1000)   # в мс
    t_overhead_list.append(t_overhead * 1000)

    # Очистка
    del out, tok
    torch.cuda.empty_cache()

Inference: 100%|██████████| 1038/1038 [09:38<00:00,  1.80it/s]


In [ ]:
# ========== Сохранение результатов ==========
# Исходный файл уже загружен в df_private
# Создаём копию и добавляем предсказания
df_result = df_private.copy()
df_result["predict_proba"] = pred_probas

OUTPUT_RESULT_CSV = "../data/bench/knowledge_bench_private_scores.csv"
df_result.to_csv(OUTPUT_RESULT_CSV, index=False)
print(f"\nРезультаты сохранены в {OUTPUT_RESULT_CSV}")

# ========== Статистика времени ==========
print("\n===== Timing (per sample) =====")
print(f"  model forward (не учитывается): {np.mean(t_model_list):.1f} ms (mean), max {np.max(t_model_list):.1f} ms")
print(f"  overhead (XGBoost + feature prep): {np.mean(t_overhead_list):.2f} ms (mean), max {np.max(t_overhead_list):.2f} ms")
print(f"  overhead p99: {np.percentile(t_overhead_list, 99):.2f} ms")
overhead_ok = np.percentile(t_overhead_list, 99) < 500
print(f"  {'✅ PASS' if overhead_ok else '❌ FAIL'}: p99 overhead < 500 ms")


Результаты сохранены в /kaggle/working/knowledge_bench_private_scores.csv

===== Timing (per sample) =====
  model forward (не учитывается): 483.5 ms (mean), max 1509.6 ms
  overhead (XGBoost + feature prep): 63.32 ms (mean), max 190.97 ms
  overhead p99: 127.65 ms
  ✅ PASS: p99 overhead < 500 ms
